# MedBhasha — COMSYS Hackathon-8 · Round 2
## Complete reproducible pipeline: NER → Clustering → Analysis → RAG → QA/UI

This notebook is organised into seven parts and preserves the original architecture:
1. Data loading
2. Tokenisation with character offsets
3. NER (BanglaBERT primary, CRF baseline/fallback)
4. Phenotype clustering from NER-derived features
5. Frequency and relationship analysis
6. RAG over the team's own annotated corpus only
7. Bengali QA and an interactive dropdown UI for the fixed queries

**Expected data files**
- `annotated_corpus.csv`: `note_id, token, tag`
- hidden test notes CSV: `note_id, note_text`
- `test_queries_bn.csv`: `query_id, query_text`

Place these files in `data/` or the notebook's working directory. Run all cells from top to bottom.

**RAG compliance:** The RAG index contains the team's own annotated corpus only. Hidden test notes are used for NER predictions and clustering, not as a RAG knowledge source. The UI calls the same retrieval function used to generate the saved query outputs.


In [3]:
# ---- 0. Setup ---------------------------------------------------------------
import os, re, json, random, math, unicodedata, zipfile, glob, shutil, warnings, collections, itertools, sys, time
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

TEAM_NAME   = "Train Test Panic"                                   # <- change before packaging
SEED        = 42
BACKEND     = os.environ.get("MB_BACKEND", "banglabert")   # 'banglabert' | 'crf'
MODEL_NAME  = os.environ.get("MB_MODEL", "csebuetnlp/banglabert")
EPOCHS      = int(os.environ.get("MB_EPOCHS", 25))         # max epochs for dev run
BATCH_SIZE  = int(os.environ.get("MB_BATCH", 16))
LR          = float(os.environ.get("MB_LR", 5e-5))
MAX_WORDS   = 90                                           # words per training/inference chunk

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except Exception: pass
set_seed()

# ---- locate repo root and data files ---------------------------------------
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if os.environ.get("MB_ROOT"): ROOT = Path(os.environ["MB_ROOT"])
PRED = ROOT / "predictions"; ANA = PRED / "analysis"
ANA.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [ROOT, ROOT/"data", ROOT/"round2_release", HERE, Path("/content"), Path("/kaggle/input"), Path("/mnt/user-data/uploads")]
def find_file(*patterns):
    for d in SEARCH_DIRS:
        if not d.exists(): continue
        for p in patterns:
            hits = sorted(glob.glob(str(d / "**" / p), recursive=True))
            if hits: return hits[0]
    raise FileNotFoundError(f"Could not find any of {patterns}. Put the CSVs in {ROOT/'data'}")
OWN_PATH   = find_file("annotated_corpus*.csv")
TEST_PATH  = find_file("/content/medbhasha_bengali_clinical_notes(test data) (1).csv")
QUERY_PATH = find_file("test_queries_bn*.csv")
print("own corpus :", OWN_PATH, "\ntest notes :", TEST_PATH, "\nqueries    :", QUERY_PATH, "\nbackend    :", BACKEND)

own corpus : /content/annotated_corpus.csv 
test notes : /content/medbhasha_bengali_clinical_notes(test data) (1).csv 
queries    : /content/test_queries_bn.csv 
backend    : banglabert


## PART 1 — DATA LOADING
Load and validate the Round-1 annotated corpus, hidden test notes, and fixed Bengali queries.

In [4]:
own_df  = pd.read_csv(OWN_PATH,  encoding="utf-8-sig", keep_default_na=False, dtype=str)
test_df = pd.read_csv(TEST_PATH, encoding="utf-8-sig", keep_default_na=False, dtype=str)
qry_df  = pd.read_csv(QUERY_PATH, encoding="utf-8-sig", keep_default_na=False, dtype=str)
assert list(test_df.columns) == ["note_id", "note_text"] and test_df.note_id.is_unique
own_df["tag"] = own_df["tag"].str.strip()
ENTITY_TYPES = sorted({t[2:] for t in own_df.tag.unique() if t != "O"})
LABELS = ["O"] + [f"{p}-{t}" for t in ENTITY_TYPES for p in ("B", "I")]
L2I = {l: i for i, l in enumerate(LABELS)}
print(len(own_df), "annotated tokens in", own_df.note_id.nunique(), "notes |", len(test_df), "test notes |", len(qry_df), "queries")
print("entity types:", ENTITY_TYPES)
own_docs = {nid: (g.token.tolist(), g.tag.tolist()) for nid, g in own_df.groupby("note_id", sort=False)}
print(own_df.tag.value_counts().to_string())

14452 annotated tokens in 300 notes | 1691 test notes | 30 queries
entity types: ['BODY_SITE', 'DIAGNOSIS', 'DOSAGE', 'MEDICATION', 'NEGATION_UNCERTAINTY', 'PROCEDURE', 'SYMPTOM', 'TEMPORAL', 'TEST', 'TEST_RESULT']
tag
O                         7592
B-SYMPTOM                  763
I-TEMPORAL                 739
I-DOSAGE                   666
B-TEST                     516
I-SYMPTOM                  467
B-TEST_RESULT              445
B-TEMPORAL                 442
B-MEDICATION               341
I-TEST_RESULT              333
B-DIAGNOSIS                331
I-TEST                     306
B-NEGATION_UNCERTAINTY     302
B-DOSAGE                   283
I-DIAGNOSIS                236
B-BODY_SITE                190
B-PROCEDURE                162
I-BODY_SITE                130
I-PROCEDURE                102
I-MEDICATION                70
I-NEGATION_UNCERTAINTY      36


## PART 2 — TOKENIZATION
Tokenize notes with character offsets and preserve BIO alignment.

In [5]:
PUNCT = "।,:;()?!\\[\\]|—–"
TOK_RE  = re.compile(rf"[{PUNCT}]|[^\s{PUNCT}]+")
ABBR_RE = re.compile(r"^[ঀ-৿A-Za-z]+(\.[ঀ-৿A-Za-z]+)+\.$")   # e.g. মি.গ্রা.
def tokenize(text):
    out = []
    for m in TOK_RE.finditer(text):
        tok, s, e = m.group(), m.start(), m.end()
        if len(tok) > 1 and tok.endswith(".") and not ABBR_RE.match(tok):
            while len(tok) > 1 and tok.endswith("."): tok, e = tok[:-1], e - 1
        out.append((tok, s, e))
    return out

# sanity: re-tokenising the own corpus text reproduces the gold tokens
agree = sum([t for t, _, _ in tokenize(" ".join(toks))] == toks for toks, _ in own_docs.values())
print(f"tokeniser reproduces gold tokenisation for {agree}/{len(own_docs)} own notes")

def chunk_tokens(n_tok_list, max_words=MAX_WORDS):
    """indices of tokens grouped into chunks <= max_words, preferring to break after '।'."""
    chunks, cur = [], []
    for i, tk in enumerate(n_tok_list):
        cur.append(i)
        if (tk == "।" and len(cur) >= max_words * 0.6) or len(cur) >= max_words:
            chunks.append(cur); cur = []
    if cur: chunks.append(cur)
    return chunks

def tags_to_spans(tags):
    """BIO tags -> list of (first_idx, last_idx, type); repairs stray I- tags."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        if t == "O" or t is None:
            if cur: spans.append(tuple(cur)); cur = None
            continue
        p, ty = t.split("-", 1)
        if p == "B" or cur is None or cur[2] != ty:
            if cur: spans.append(tuple(cur))
            cur = [i, i, ty]
        else:
            cur[1] = i
    if cur: spans.append(tuple(cur))
    return spans

tokeniser reproduces gold tokenisation for 296/300 own notes


## PART 3 — NER
Train/evaluate the CRF baseline and BanglaBERT token classifier; retrain on all annotated notes and predict hidden test notes.

In [6]:
!pip install -q seqeval
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
!pip install -q sklearn-crfsuite
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
rng = random.Random(SEED)
by_prefix = collections.defaultdict(list)
for nid in own_docs: by_prefix[nid[0]].append(nid)
dev_ids = []
for p, ids in sorted(by_prefix.items()):
    ids = sorted(ids); rng.shuffle(ids); dev_ids += ids[:max(1, round(0.15 * len(ids)))]
dev_ids = sorted(dev_ids); train_ids = sorted(set(own_docs) - set(dev_ids))
print(len(train_ids), "train notes /", len(dev_ids), "dev notes")

def make_chunks(ids):
    X, Y = [], []
    for nid in ids:
        toks, tags = own_docs[nid]
        for idx in chunk_tokens(toks):
            X.append([toks[i] for i in idx]); Y.append([tags[i] for i in idx])
    return X, Y
Xtr, Ytr = make_chunks(train_ids); Xdv, Ydv = make_chunks(dev_ids)
Xall, Yall = make_chunks(sorted(own_docs))

# ---- CRF baseline ----------------------------------------------------------
import sklearn_crfsuite
def word2feat(sent, i):
    w = sent[i]; f = {"bias": 1.0, "w": w, "w3": w[-3:], "w2": w[-2:], "p3": w[:3], "p2": w[:2],
                      "isdig": w.isdigit(), "hasdig": any(c.isdigit() for c in w), "punct": w in "।,:;()?", "len": min(len(w), 12)}
    for d in (-3, -2, -1, 1, 2, 3):
        j = i + d
        if 0 <= j < len(sent): f[f"{d}:w"] = sent[j]; f[f"{d}:s3"] = sent[j][-3:]
        else: f[f"{d}:pad"] = True
    return f
def sent2feats(s): return [word2feat(s, i) for i in range(len(s))]
def train_crf(X, Y):
    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=200, all_possible_transitions=True)
    crf.fit([sent2feats(s) for s in X], Y); return crf
set_seed()
crf_dev = train_crf(Xtr, Ytr)
crf_dev_pred = crf_dev.predict([sent2feats(s) for s in Xdv])
crf_scores = dict(P=precision_score(Ydv, crf_dev_pred), R=recall_score(Ydv, crf_dev_pred), F1=f1_score(Ydv, crf_dev_pred))
print("CRF baseline (dev, entity-level exact match):", {k: round(v, 4) for k, v in crf_scores.items()})

258 train notes / 42 dev notes
CRF baseline (dev, entity-level exact match): {'P': np.float64(0.8713), 'R': np.float64(0.8193), 'F1': np.float64(0.8445)}


### 3.2 BanglaBERT token classifier (fine-tuning)
First sub-word of each word carries the label; other sub-words are ignored in the loss. The csebuetnlp normaliser is applied to the *model input only*
(never to the text used for offsets). If BanglaBERT cannot be downloaded the notebook falls back to the CRF and says so loudly.

In [8]:
USED_BACKEND = BACKEND
bert = {}
if BACKEND == "banglabert":
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
        try:
            from normalizer import normalize as _bn_norm
        except Exception:
            try:
                import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/csebuetnlp/normalizer"], check=True, timeout=300)
                from normalizer import normalize as _bn_norm
            except Exception:
                _bn_norm = None; print("! csebuetnlp normalizer unavailable - continuing without it")
        hf_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        assert hf_tok.is_fast, "fast tokenizer required for word_ids()"
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        print("BanglaBERT tokenizer loaded; device =", DEVICE)
    except Exception as ex:
        print("!!! BanglaBERT unavailable ->", repr(ex)[:200], "\n!!! FALLING BACK TO CRF BACKEND"); USED_BACKEND = "crf"

def _prep(w):
    if _bn_norm is None: return w
    try:
        n = _bn_norm(w); return n if n.strip() else w
    except Exception: return w

def encode(words, labels=None):
    enc = hf_tok([_prep(w) for w in words], is_split_into_words=True, truncation=True, max_length=512)
    wid = enc.word_ids(); lab = []; prev = None
    for k in wid:
        if k is None or k == prev: lab.append(-100)
        else: lab.append(L2I[labels[k]] if labels is not None else 0)
        prev = k
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": lab, "word_ids": wid}

def collate(items):
    m = max(len(x["input_ids"]) for x in items); pad = hf_tok.pad_token_id
    return {"input_ids": torch.tensor([x["input_ids"] + [pad]*(m-len(x["input_ids"])) for x in items]),
            "attention_mask": torch.tensor([x["attention_mask"] + [0]*(m-len(x["attention_mask"])) for x in items]),
            "labels": torch.tensor([x["labels"] + [-100]*(m-len(x["labels"])) for x in items])}

def bert_predict(model, word_lists, bs=32):
    model.eval(); out = []
    encs = [encode(w) for w in word_lists]
    with torch.no_grad():
        for i in range(0, len(encs), bs):
            batch = encs[i:i+bs]; b = collate(batch)
            logits = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits.argmax(-1).cpu().tolist()
            for e, lg, words in zip(batch, logits, word_lists[i:i+bs]):
                tags = ["O"] * len(words); seen = set()
                for pos, k in enumerate(e["word_ids"]):
                    if k is not None and k not in seen: tags[k] = LABELS[lg[pos]]; seen.add(k)
                out.append(tags)
    return out

def train_bert(X, Y, epochs, dev=None, tag=""):
    set_seed()
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I, ignore_mismatched_sizes=True).to(DEVICE)
    data = [encode(w, l) for w, l in zip(X, Y)]
    steps = epochs * math.ceil(len(data) / BATCH_SIZE)
    nd = ["bias", "LayerNorm.weight"]
    params = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nd)], "weight_decay": 0.01},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in nd)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=LR)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    g = torch.Generator(); g.manual_seed(SEED); hist = []; best = (-1, 0, None)
    for ep in range(1, epochs + 1):
        model.train(); order = torch.randperm(len(data), generator=g).tolist(); tot = 0
        for i in range(0, len(order), BATCH_SIZE):
            b = collate([data[j] for j in order[i:i+BATCH_SIZE]]); b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = model(**b).loss; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad(); tot += loss.item()
        if dev is not None:
            pred = bert_predict(model, dev[0]); f1 = f1_score(dev[1], pred); hist.append((ep, tot, f1))
            print(f"{tag} epoch {ep:02d} loss {tot:.3f} dev-F1 {f1:.4f}")
            if f1 > best[0]: best = (f1, ep, pred)
    return model, hist, best

bert_scores, best_epochs, ner_dev_report = None, None, ""
if USED_BACKEND == "banglabert":
    _, hist, (bf1, best_epochs, bpred) = train_bert(Xtr, Ytr, EPOCHS, dev=(Xdv, Ydv), tag="dev")
    bert_scores = dict(P=precision_score(Ydv, bpred), R=recall_score(Ydv, bpred), F1=bf1)
    ner_dev_report = classification_report(Ydv, bpred, digits=4)
    print(f"\nBanglaBERT dev (best epoch {best_epochs}):", {k: round(v, 4) for k, v in bert_scores.items()}); print(ner_dev_report)
    pd.DataFrame(hist, columns=["epoch", "train_loss", "dev_f1"]).to_csv(ANA / "ner_dev_training_curve.csv", index=False)
else:
    ner_dev_report = classification_report(Ydv, crf_dev_pred, digits=4); print(ner_dev_report)
(ANA / "ner_dev_report.txt").write_text(
    f"backend={USED_BACKEND}\nCRF baseline dev: {crf_scores}\nBanglaBERT dev: {bert_scores} (best epoch {best_epochs})\n\n{ner_dev_report}", encoding="utf-8")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

BanglaBERT tokenizer loaded; device = cuda


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

dev epoch 01 loss 51.227 dev-F1 0.0272
dev epoch 02 loss 31.691 dev-F1 0.0817
dev epoch 03 loss 18.970 dev-F1 0.5182
dev epoch 04 loss 11.216 dev-F1 0.7577
dev epoch 05 loss 7.040 dev-F1 0.8274
dev epoch 06 loss 4.571 dev-F1 0.8655
dev epoch 07 loss 3.198 dev-F1 0.8928
dev epoch 08 loss 2.224 dev-F1 0.8960
dev epoch 09 loss 1.768 dev-F1 0.8767
dev epoch 10 loss 1.320 dev-F1 0.8860
dev epoch 11 loss 1.065 dev-F1 0.8793
dev epoch 12 loss 0.874 dev-F1 0.8945
dev epoch 13 loss 0.679 dev-F1 0.8970
dev epoch 14 loss 0.603 dev-F1 0.9010
dev epoch 15 loss 0.558 dev-F1 0.9029
dev epoch 16 loss 0.473 dev-F1 0.8970
dev epoch 17 loss 0.447 dev-F1 0.9001
dev epoch 18 loss 0.381 dev-F1 0.9042
dev epoch 19 loss 0.368 dev-F1 0.9048
dev epoch 20 loss 0.356 dev-F1 0.9055
dev epoch 21 loss 0.331 dev-F1 0.9007
dev epoch 22 loss 0.326 dev-F1 0.9067
dev epoch 23 loss 0.319 dev-F1 0.9077
dev epoch 24 loss 0.303 dev-F1 0.9066
dev epoch 25 loss 0.306 dev-F1 0.9067

BanglaBERT dev (best epoch 23): {'P': np.floa

1164

### 3.3 Final model on all 300 annotated notes → predict every test note
Entities are mapped back to character offsets of the **unmodified** `note_text`.

In [9]:
if USED_BACKEND == "banglabert":
    final_model, _, _ = train_bert(Xall, Yall, max(8, best_epochs), tag="final")
else:
    final_crf = train_crf(Xall, Yall)

def predict_note(text):
    toks = tokenize(text); words = [t for t, _, _ in toks]; tags = ["O"] * len(words)
    chunks = chunk_tokens(words); lists = [[words[i] for i in c] for c in chunks]
    preds = bert_predict(final_model, lists) if USED_BACKEND == "banglabert" else final_crf.predict([sent2feats(s) for s in lists])
    for c, p in zip(chunks, preds):
        for i, t in zip(c, p): tags[i] = t
    rows = []
    for a, b, ty in tags_to_spans(tags):
        if toks[a][0] in "।,:;()?!" : continue
        rows.append((toks[a][1], toks[b][2], ty))
    return rows

set_seed(); recs = []
for nid, text in zip(test_df.note_id, test_df.note_text):
    for s, e, ty in predict_note(text):
        recs.append((nid, s, e, ty, text[s:e]))
ner_pred = pd.DataFrame(recs, columns=["note_id", "start", "end", "entity_type", "entity_text"])
assert ner_pred.entity_type.isin(ENTITY_TYPES).all()
assert all(test_df.set_index("note_id").note_text[r.note_id][r.start:r.end] == r.entity_text for r in ner_pred.itertuples())
ner_pred.to_csv(PRED / "ner_predictions.csv", index=False, encoding="utf-8")
print(len(ner_pred), "entities over", ner_pred.note_id.nunique(), "of", len(test_df), "notes")
print(ner_pred.entity_type.value_counts().to_string()); ner_pred.head(10)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


50686 entities over 1691 of 1691 notes
entity_type
SYMPTOM                 11766
TEST_RESULT              6405
TEMPORAL                 5721
BODY_SITE                5366
TEST                     5251
MEDICATION               4882
DOSAGE                   4705
DIAGNOSIS                2814
NEGATION_UNCERTAINTY     2427
PROCEDURE                1349


,note_id,start,end,entity_type,entity_text
0,MB-0001,137,146,TEMPORAL,২৫ সপ্তাহ
1,MB-0001,179,188,SYMPTOM,মদ্যপানের
2,MB-0001,196,199,NEGATION_UNCERTAINTY,নেই
3,MB-0001,222,229,TEST,রক্তচাপ
4,MB-0001,230,248,TEST_RESULT,১১০/৭০ মি.মি. পারদ
5,MB-0001,250,264,TEST,ফান্ডাল উচ্চতা
6,MB-0001,297,315,TEST,ভ্রূণের হৃদস্পন্দন
7,MB-0001,316,325,TEST_RESULT,১৪০/মিনিট
8,MB-0001,382,391,TEMPORAL,২৫ সপ্তাহ
9,MB-0001,407,412,MEDICATION,আয়রন


## PART 4 — PHENOTYPE CLUSTERING
Cluster hidden test notes using features derived from NER predictions only.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# Bengali-capable font for charts (best effort; CSV tables are the primary analysis output)
def _setup_font():
    for f in fm.fontManager.ttflist:
        if any(k in f.name for k in ("Bengali", "Siyam", "Kalpurush", "Nirmala", "Lohit-Bengali", "Vrinda")):
            plt.rcParams["font.family"] = f.name; return f.name
    try:
        import urllib.request
        p = "/tmp/NotoSansBengali.ttf"
        urllib.request.urlretrieve("https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSansBengali/hinted/ttf/NotoSansBengali-Regular.ttf", p)
        fm.fontManager.addfont(p); plt.rcParams["font.family"] = fm.FontProperties(fname=p).get_name(); return "NotoSansBengali"
    except Exception: return None
print("chart font:", _setup_font())

def bn_digits(s): return str(s).translate(str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789"))
def norm_ent(s): return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s).replace("‌", "").replace("‍", "")).strip().strip(".")

ent = ner_pred.copy(); ent["e"] = ent.entity_text.map(norm_ent)
ent = ent[ent.e.str.len() > 1]
FEAT_TYPES = {"DIAGNOSIS": 3.0, "MEDICATION": 2.0, "SYMPTOM": 1.5, "PROCEDURE": 1.0, "TEST": 1.0, "BODY_SITE": 0.7}
docs = {nid: [] for nid in test_df.note_id}
for r in ent.itertuples():
    if r.entity_type in FEAT_TYPES:
        docs[r.note_id].append(f"{r.entity_type}|{r.e}".replace(" ", "_"))
        docs[r.note_id] += [f"{r.entity_type}~{w}" for w in r.e.split()]
note_ids = list(test_df.note_id)
vec = TfidfVectorizer(analyzer=lambda d: d, sublinear_tf=True, min_df=2)
Xtf = vec.fit_transform([docs[n] for n in note_ids])
w = np.array([FEAT_TYPES[f.split("|")[0].split("~")[0]] for f in vec.get_feature_names_out()])
Xw = Xtf.multiply(w).tocsr()
nsvd = max(2, min(100, Xw.shape[1] - 1, Xw.shape[0] - 1))
Z = Normalizer().fit_transform(TruncatedSVD(nsvd, random_state=SEED).fit_transform(Xw))
print("feature matrix", Xw.shape, "-> SVD", Z.shape)

K_MIN, K_MAX, K_STEP, PARSIMONY = 20, 120, 2, 0.95
rows = []
for k in range(K_MIN, K_MAX + 1, K_STEP):
    lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)
    rows.append((k, silhouette_score(Z, lab, metric="cosine"), davies_bouldin_score(Z, lab)))
sel = pd.DataFrame(rows, columns=["k", "silhouette", "davies_bouldin"]); sel.to_csv(ANA / "cluster_k_selection.csv", index=False)
# parsimony rule: smallest k whose Silhouette is within 5% of the best (avoids over-splitting near-duplicate templated notes)
BEST_K = int(sel.loc[sel.silhouette >= PARSIMONY * sel.silhouette.max(), "k"].min())
km = KMeans(BEST_K, n_init=20, random_state=SEED).fit(Z); labels = km.labels_
SIL = silhouette_score(Z, labels, metric="cosine"); DBI = davies_bouldin_score(Z, labels)
print(f"chosen k={BEST_K}  Silhouette={SIL:.4f}  Davies-Bouldin={DBI:.4f}")
clusters = pd.DataFrame({"note_id": note_ids, "cluster_id": labels}); clusters.to_csv(PRED / "clusters.csv", index=False)
assert len(clusters) == len(test_df) and clusters.note_id.is_unique
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5)); ax[0].plot(sel.k, sel.silhouette); ax[0].axvline(BEST_K, ls="--", c="r"); ax[0].set_title("Silhouette vs k")
ax[1].plot(sel.k, sel.davies_bouldin); ax[1].axvline(BEST_K, ls="--", c="r"); ax[1].set_title("Davies-Bouldin vs k"); plt.tight_layout(); plt.savefig(ANA / "cluster_k_selection.png", dpi=130); plt.close()

chart font: NotoSansBengali
feature matrix (1691, 3680) -> SVD (1691, 100)
chosen k=84  Silhouette=0.8465  Davies-Bouldin=0.6841


## PART 5 — FREQUENCY AND RELATIONSHIP ANALYSIS
Entity frequency, type co-occurrence, diagnosis associations, heatmap, network and cluster profiles.

In [11]:
# Frequency tables / charts
freq = ent.groupby(["entity_type", "e"]).agg(count=("note_id", "size"), notes=("note_id", "nunique")).reset_index().sort_values(["entity_type", "count"], ascending=[True, False])
freq.to_csv(ANA / "entity_frequency_all.csv", index=False, encoding="utf-8")
ent.entity_type.value_counts().rename_axis("entity_type").reset_index(name="count").to_csv(ANA / "entity_type_counts.csv", index=False)
for ty in ("DIAGNOSIS", "MEDICATION", "SYMPTOM", "TEST", "PROCEDURE", "BODY_SITE"):
    top = freq[freq.entity_type == ty].head(25)
    top.to_csv(ANA / f"top_{ty.lower()}.csv", index=False, encoding="utf-8")
    if len(top):
        plt.figure(figsize=(7, 6)); plt.barh(top.e[::-1], top.notes[::-1]); plt.title(f"Top {ty} (notes)"); plt.tight_layout(); plt.savefig(ANA / f"top_{ty.lower()}.png", dpi=120); plt.close()
plt.figure(figsize=(7, 3.5)); ent.entity_type.value_counts().plot.bar(); plt.title("Predicted entities per type"); plt.tight_layout(); plt.savefig(ANA / "entity_type_counts.png", dpi=120); plt.close()

# Relationship analysis 1: type-type co-occurrence (notes containing both)
pres = pd.crosstab(ent.note_id, ent.entity_type).clip(upper=1)
cooc = pres.T.dot(pres); cooc.to_csv(ANA / "type_cooccurrence.csv")
plt.figure(figsize=(6, 5)); plt.imshow(cooc.values, cmap="Blues"); plt.xticks(range(len(cooc)), cooc.columns, rotation=90); plt.yticks(range(len(cooc)), cooc.index)
plt.colorbar(); plt.title("Entity-type co-occurrence (notes)"); plt.tight_layout(); plt.savefig(ANA / "type_cooccurrence.png", dpi=120); plt.close()

# Relationship analysis 2: diagnosis <-> medication / symptom / test association (support, lift, PMI)
def assoc(typeA, typeB, min_support=5, topn=60):
    A = ent[ent.entity_type == typeA].drop_duplicates(["note_id", "e"]); B = ent[ent.entity_type == typeB].drop_duplicates(["note_id", "e"])
    N = len(test_df); ca, cb = A.e.value_counts(), B.e.value_counts()
    m = A.merge(B, on="note_id", suffixes=("_a", "_b")).groupby(["e_a", "e_b"]).size().reset_index(name="support")
    m = m[m.support >= min_support]
    m["lift"] = m.support * N / (m.e_a.map(ca) * m.e_b.map(cb)); m["pmi"] = np.log2(m.lift)
    return m.sort_values(["lift", "support"], ascending=False).head(topn).rename(columns={"e_a": typeA.lower(), "e_b": typeB.lower()})
for a, b in [("DIAGNOSIS", "MEDICATION"), ("DIAGNOSIS", "SYMPTOM"), ("DIAGNOSIS", "TEST")]:
    t = assoc(a, b); t.to_csv(ANA / f"assoc_{a.lower()}_{b.lower()}.csv", index=False, encoding="utf-8"); print(a, "<->", b, len(t), "pairs")

# Heat-map: top diagnoses x top medications
topd = freq[freq.entity_type == "DIAGNOSIS"].head(20).e.tolist(); topm = freq[freq.entity_type == "MEDICATION"].head(20).e.tolist()
D = ent[(ent.entity_type == "DIAGNOSIS") & ent.e.isin(topd)].drop_duplicates(["note_id", "e"]); M = ent[(ent.entity_type == "MEDICATION") & ent.e.isin(topm)].drop_duplicates(["note_id", "e"])
hm = D.merge(M, on="note_id").groupby(["e_x", "e_y"]).size().unstack(fill_value=0).reindex(index=topd, columns=topm, fill_value=0)
hm.to_csv(ANA / "diagnosis_x_medication_matrix.csv", encoding="utf-8")
plt.figure(figsize=(11, 8)); plt.imshow(hm.values, cmap="YlOrRd"); plt.xticks(range(len(topm)), topm, rotation=90, fontsize=7); plt.yticks(range(len(topd)), topd, fontsize=7)
plt.colorbar(); plt.title("Diagnosis x Medication co-occurrence (notes)"); plt.tight_layout(); plt.savefig(ANA / "diagnosis_x_medication_heatmap.png", dpi=130); plt.close()

# Relationship analysis 3: entity network
import networkx as nx
G = nx.Graph(); pairs = pd.concat([assoc("DIAGNOSIS", "MEDICATION", 8, 40), assoc("DIAGNOSIS", "SYMPTOM", 8, 40)])
for r in pairs.itertuples(index=False):
    a, b = r[0], r[1]; G.add_node(a, t="D"); G.add_node(b, t="O"); G.add_edge(a, b, weight=r.support)
if G.number_of_edges():
    plt.figure(figsize=(11, 9)); pos = nx.spring_layout(G, seed=SEED, k=0.6)
    nx.draw_networkx_edges(G, pos, alpha=.3); nx.draw_networkx_nodes(G, pos, node_size=[300 if G.nodes[n]["t"] == "D" else 90 for n in G], node_color=["tab:red" if G.nodes[n]["t"] == "D" else "tab:blue" for n in G], alpha=.8)
    nx.draw_networkx_labels(G, pos, font_size=6); plt.axis("off"); plt.title("Diagnosis (red) - medication/symptom (blue) network"); plt.savefig(ANA / "entity_network.png", dpi=130); plt.close()

# Cluster profiles
prof = []
for c in range(BEST_K):
    ids = clusters.note_id[clusters.cluster_id == c]; sub = ent[ent.note_id.isin(ids)]
    top = lambda ty, n=3: "; ".join(f"{k}({v})" for k, v in sub[sub.entity_type == ty].drop_duplicates(["note_id", "e"]).e.value_counts().head(n).items())
    prof.append((c, len(ids), top("DIAGNOSIS"), top("MEDICATION", 4), top("SYMPTOM", 4), top("TEST", 3)))
prof = pd.DataFrame(prof, columns=["cluster_id", "n_notes", "top_diagnoses", "top_medications", "top_symptoms", "top_tests"])
prof.to_csv(ANA / "cluster_profiles.csv", index=False, encoding="utf-8")
(ANA / "cluster_metrics.txt").write_text(f"k={BEST_K}\nsilhouette(cosine)={SIL:.4f}\ndavies_bouldin={DBI:.4f}\nfeatures={Xw.shape}\nsvd_dims={nsvd}\n")
plt.figure(figsize=(8, 3.5)); clusters.cluster_id.value_counts().sort_index().plot.bar(); plt.title("Cluster sizes"); plt.tight_layout(); plt.savefig(ANA / "cluster_sizes.png", dpi=120); plt.close()
prof.head(12)

DIAGNOSIS <-> MEDICATION 60 pairs
DIAGNOSIS <-> SYMPTOM 60 pairs
DIAGNOSIS <-> TEST 60 pairs


,cluster_id,n_notes,top_diagnoses,top_medications,top_symptoms,top_tests
0,0,21,প্লান্টার ফ্যাসাইটিস(10); হিল পেইন(9); ফ্যাসিয...,ডাইক্লোফেনাক জেল(21); প্যারাসিটামল(21); সিলিকন...,ব্যথা(21); ফোলা(16); লালচে ভাব(16); তীব্র(4),এক্স-রে(12); হিল স্পার(10); শারীরিক পরীক্ষা(2)
1,1,43,তীব্র আর্টিকারিয়া(19); আর্টিকারিয়া(15); আমবা...,ক্যালামাইন লোশন(27); সিটিরিজিন(15); প্রেডনিসোল...,চুলকাচ্ছে(43); শ্বাসকষ্ট(36); র্যাশ(31); ফুলে(31),ওজন(15); ডার্মোগ্রাফিজম(14); নাড়ি(10)
2,2,25,গাউট(14); তীব্র গাউটি আর্থ্রাইটিস(14); ইউরিক অ...,কলচিসিন(13); ইটোরিকক্সিব(13); প্যান্টোপ্রাজল(1...,ফোলা(25); তীব্র ব্যথা(24); ব্যথা(20); ফুলে(17),মেটাটারসোফ্যালানজিয়াল(22); সিরাম ইউরিক অ্যাসি...
3,3,19,টনসিলাইটিস(12); তীব্র টনসিলোফ্যারিঞ্জাইটিস(7);...,প্যারাসিটামল সিরাপ(17); অ্যামোক্সিসিলিন(15); অ...,জ্বর(19); গলায় ব্যথা(19); গায়ে পুঁজ(19); ফোল...,তাপমাত্রা(19); নাড়ির গতি(19); ওজন(19)
4,4,35,স্ক্যাবিস(32); খোসপাঁচড়া(2); স্ক্যাবিস)(1),পারমেথ্রিন(24); সিটিরিজিন(24); ক্যালামাইন লোশন...,চুলকানি(32); ফুসকুড়ি(32); খোসপাঁচড়া(18); পুঁ...,ওজন(13); শারীরিক পরীক্ষা(7); ভাইটাল সাইন(3)
5,5,43,অ্যালার্জিক রাইনাইটিস(22); মৌসুমি অ্যালার্জিক ...,স্যালাইন নাসাল ড্রপ(15); ফেক্সোফেনাডিন(15); ফ্...,ফোলা(43); হাঁচি(35); শ্লেষ্মা(34); কালচে ছোপ(26),ওজন(15); শারীরিক পরীক্ষা(10); শ্লেষ্মা(6)
6,6,21,কর্নিয়াল অ্যাব্রেশন(9); ডায়াবেটিস(3); উচ্চ র...,মক্সিফ্লক্সাসিন(21); আই ড্রপ(21); লুব্রিকেন্ট ...,জ্বালা করছে(21); চোখ কচলাবেন(16); চোখে ধুলো পড...,দৃষ্টিশক্তি(17); কনজাংটিভা(16); ফ্লুরেসিন স্টে...
7,7,26,গ্যাস্ট্রাইটিস(9); অ্যাসিডিটি(6); epigastrium(5),epigastrium(16); NSAID(11); ওমিপ্রাজল(8); ডমপে...,জ্বালাভাব(26); ব্যথা(17); বুক জ্বালা(15); পেট ...,শব্দ(16); শারীরিক পরীক্ষা(6); পেটের আলট্রাসনোগ...
8,8,15,হাঁপানি(10); ব্রংকিয়াল অ্যাজমা(5); শিশুর(1),সালবিউটামল ইনহেলার স্পেসারসহ(15); বুডেসোনাইড ই...,কাশি(15); শ্বাসকষ্ট(15); সাঁ সাঁ শব্দ(12); ঠোঁ...,তাপমাত্রা(15); নাড়ির গতি(15); ওজন(15)
9,9,28,ভাইরাল জ্বর(9); জল(3),প্যারাসিটামল(28); নাকের স্যালাইন ড্রপ(28),জ্বর(28); শ্বাসকষ্ট(19); খিঁচুনি(19); সর্দি-কা...,তাপমাত্রা(28); নাড়ির গতি(28); ওজন(28)


## PART 6 — RAG
Retrieve and answer using the team's own annotated corpus only. Hidden test notes are excluded from the RAG knowledge base.

In [12]:
from sklearn.metrics.pairwise import linear_kernel
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789"); EN2BN = str.maketrans("0123456789", "০১২৩৪৫৬৭৮৯")
def nrm(s):
    s = unicodedata.normalize("NFC", s).replace("‌", "").replace("‍", "").translate(BN2EN)
    return s.replace("ী", "ি").replace("ূ", "ু")
def bn(n): return str(n).translate(EN2BN)
L = r"(?<![ঀ-৿])"          # left word boundary for Bengali

CONCEPTS = {}
def C(key, kind, pat, trig, label, sent=False):
    CONCEPTS[key] = dict(sent=sent, kind=kind, rx=re.compile(L + "(?:" + nrm(pat) + ")", re.I), trig=re.compile(nrm(trig), re.I) if trig else None, label=label)
ABX = r"(?:অ্যান্টিবায়োটিক|অ্যামোক্সিসিলিন|সেফিক্সিম|সেফট্রিয়াক্সোন|সেফুরক্সিম|সেফালেক্সিন|সেফপোডক্সিম|অ্যাজিথ্রোমাইসিন|ক্ল্যারিথ্রোমাইসিন|সিপ্রোফ্লক্সাসিন|লেভোফ্লক্সাসিন|অফলক্সাসিন|নাইট্রোফিউরানটোইন|ফসফোমাইসিন|কো-?ট্রাইমোক্সাজল|ডক্সিসাইক্লিন|মেট্রোনিডাজল|ক্লিন্ডামাইসিন|জেন্টামাইসিন|ক্লাভুলানিক|অ্যামিকাসিন|পিপারাসিলিন|মেরোপেনেম|ফ্লুক্লক্সাসিলিন|নরফ্লক্সাসিন|ফিউসিডিক|মুপিরোসিন|কোট্রিমোক্সাজল|সেফ\S+|\S*সিলিন|\S*মাইসিন|\S*ফ্লক্সাসিন)"
# --- diagnoses / conditions ('dx': patient has it; negated, family-history and advice mentions are ignored)
C("DM", "dx", r"ডায়াবেটিস|ডায়াবেটিক|টাইপ-?২|T2DM|T1DM|diabet", r"ডায়াবেটিস|ডায়াবেটিক|T2DM", "ডায়াবেটিস")
C("HTN", "dx", r"উচ্চ রক্তচাপ|হাইপারটেনশন|হাইপারটেনসিভ|hypertension", r"উচ্চ রক্তচাপ", "উচ্চ রক্তচাপ")
C("PNEUMONIA", "dx", r"নিউমোনিয়া|pneumonia", r"নিউমোনিয়া", "নিউমোনিয়া")
C("RTI", "dx", r"নিউমোনিয়া|শ্বাসতন্ত্রের সংক্রমণ|শ্বাসনালির সংক্রমণ|ফুসফুসের সংক্রমণ|ব্রঙ্কাইটিস|ব্রংকাইটিস|ব্রঙ্কিওলাইটিস", r"শ্বাসতন্ত্রের সংক্রমণ", "শ্বাসতন্ত্রের সংক্রমণ")
C("HF", "dx", r"হার্ট ?ফে(ই)?লি?(ও|উ)র|হৃদযন্ত্রের (বিকলতা|ব্যর্থতা)|হৃদ্?যন্ত্র বিকল|কনজেস্টিভ (কার্ডিয়াক|হার্ট)|heart failure|\bCHF\b|\bCCF\b", r"হার্ট ফেইলিওর|হার্ট ফেলিওর", "হার্ট ফেইলিওর")
C("KIDNEY", "dx", r"কিডনি(র)? (সমস্যা|রোগ|বিকল|ফেইলি)|ক্রনিক কিডনি|\bCKD\b|বৃক্ক|কিডনি ফেইলিওর|রেনাল", r"কিডনির সমস্যা|কিডনি", "কিডনির সমস্যা")
C("ASTHMA", "dx", r"হাঁপানি|অ্যাজমা|asthma", r"হাঁপানি|অ্যাজমা", "হাঁপানি")
C("GOUT", "dx", r"গাউট|গাউটি|gout", r"গাউট", "গাউট")
C("PEPTIC", "dx", r"পেপটিক|গ্যাস্ট্রিক আলসার|ডুওডেনাল আলসার|peptic", r"পেপটিক", "পেপটিক আলসার")
C("ALLERGIC_RHINITIS", "dx", r"অ্যালার্জিক রাইনাইটিস|এলার্জিক রাইনাইটিস|allergic rhinitis", r"রাইনাইটিস", "অ্যালার্জিক রাইনাইটিস")
C("DENGUE", "dx", r"ডেঙ্গু|dengue", r"ডেঙ্গু", "ডেঙ্গু")
C("SCABIES", "dx", r"স্ক্যাবিস|খোসপাঁচড়া|scabies", r"স্ক্যাবিস|খোসপাঁচড়া", "স্ক্যাবিস")
C("DEPRESSION", "dx", r"বিষণ্ণ|বিষণ্ন|ডিপ্রেসি|ডিপ্রেশন|depress", r"বিষণ্ণ|বিষণ্ন", "বিষণ্নতা")
C("SEPSIS", "dx", r"সেপসিস|সেপ্সিস|সেপ্টিসেমিয়া|সেপটিসেমিয়া|sepsis|সেপ্টিক|সেপটিক", r"সেপসিস|সেপ্সিস", "সেপসিস")
C("HYPERCHOL", "dx", r"হাইপারকোলেস্টেরোলেমিয়া|ডিসলিপিডেমিয়া|হাইপারলিপিডেমিয়া|উচ্চ কোলেস্টেরল|কোলেস্টেরল বেশি", r"হাইপারকোলেস্টেরোলেমিয়া", "হাইপারকোলেস্টেরোলেমিয়া")
C("ISCHEMIC", "dx", r"ইস্কিমিক|ইসকেমিক|করোনারি|এনজাইনা|এনজিনা|\bACS\b|মায়োকার্ডিয়াল ইনফার্কশন|হার্ট অ্যাটাক|ischemic|coronary", r"ইস্কিমিক|করোনারি", "ইস্কিমিক/করোনারি হৃদরোগ (সন্দেহ)")
C("URTICARIA", "dx", r"তীব্র[^;।,]{0,12}(আর্টিকারিয়া|আমবাত)|অ্যাকিউট আর্টিকারিয়া|acute urticaria", r"তীব্র আর্টিকারিয়া", "তীব্র আর্টিকারিয়া")
C("VIRAL_FEVER", "dx", r"ভাইরাল[^;।,]{0,20}জ্বর|ভাইরাল সংক্রমণ|ভাইরাল ফিভার|viral fever", r"ভাইরাল জ্বর", "ভাইরাল জ্বর")
C("BPH", "dx", r"\bBPH\b|প্রস্টেট বৃদ্ধি|প্রস্টেটের বৃদ্ধি|প্রস্টেট গ্রন্থির বৃদ্ধি|প্রোস্টেট বৃদ্ধি|বেনাইন প্রস্টেটিক|প্রস্টেট বর্ধিত", r"BPH|প্রস্টেট", "প্রস্টেট বৃদ্ধি (BPH)")
C("TB", "dx", r"যক্ষ্মা|টিবি|\bTB\b(?![A-Za-z])|tuberculosis", r"যক্ষ্মা", "যক্ষ্মা (সন্দেহ)")
C("ZOSTER", "dx", r"হারপিস|হার্পিস|শিংগলস|জোস্টার|zoster", r"হারপিস|জোস্টার", "হারপিস জোস্টার")
C("MIGRAINE", "dx", r"মাইগ্রেন|migraine", r"মাইগ্রেন", "মাইগ্রেন")
C("TYPHOID", "dx", r"টাইফয়েড|এন্টেরিক ফিভার|enteric fever|typhoid", r"টাইফয়েড", "টাইফয়েড")
C("PCOS", "dx", r"\bPCOS\b|পিসিওএস|পলিসিস্টিক ওভারি", r"PCOS", "PCOS")
C("UTI", "dx", r"ইউটিআই|\bUTI\b|মূত্রনালির সংক্রমণ|মূত্রথলির সংক্রমণ|সিস্টাইটিস|পাইলোনেফ্রাইটিস", r"UTI|ইউটিআই", "ইউটিআই (UTI)")
# --- symptoms ('sym': present now; excludes negated/family/advice/conditional)
C("DYSPNEA", "sym", r"শ্বাসকষ্ট|শ্বাস কষ্ট|শ্বাসক্লেশ|হাঁপিয়ে|শ্বাস নিতে (কষ্ট|অসুবিধা)", r"শ্বাসকষ্ট", "শ্বাসকষ্ট")
C("NIGHT_COUGH", "sym", r"রাতে[^;।,]{0,30}কাশি|কাশি[^;।,]{0,25}রাতে|রাতের কাশি", r"রাতে কাশি", "রাতে কাশি")
C("DIARRHEA", "sym", r"পাতলা পায়খানা|ডায়রিয়া|diarrh", r"পাতলা পায়খানা", "পাতলা পায়খানা")
C("DEHYD", "sym", r"জল ?শূন্যতা|পানিশূন্যতা|ডিহাইড্রেশন|dehydrat", r"জল ?শূন্যতা", "জল শূন্যতা")
C("EPIG_BURN", "sym", r"(পেটের|পেটে)[^;।,]{0,30}জ্বালা|বুক ?জ্বালা|অম্বল|ঊর্ধ্ব ?পেট", r"পেটের (ওপরের|উপরের)", "পেটের ওপরের অংশে জ্বালাপোড়া")
C("SNEEZE", "sym", r"হাঁচি", r"হাঁচি", "হাঁচি")
C("FEVER", "sym", r"জ্বর|fever", r"জ্বর", "জ্বর")
C("NIGHT_ITCH", "sym", r"রাতে[^;।,]{0,30}চুলকা|চুলকা[^;।,]{0,25}রাতে", r"রাতে চুলকানি", "রাতে চুলকানি")
C("FATIGUE", "sym", r"ক্লান্ত|দুর্বল|অবসন্ন|অবসাদ", r"ক্লান্তি|দুর্বলতা", "ক্লান্তি/দুর্বলতা")
C("CONFUSION", "sym", r"বিভ্রান্ত|কনফিউশন|অসংলগ্ন|confus", r"বিভ্রান্তি|কনফিউশন", "বিভ্রান্তি")
C("CHEST_PAIN", "sym", r"বুকে (ব্যথা|ব্যাথা|চাপ|চিনচিন)|বুক ব্যথা", r"বুকে ব্যথা", "বুকে ব্যথা")
C("COLD", "sym", r"সর্দি", r"সর্দি", "সর্দি-কাশি")
C("BLISTER", "sym", r"ফোসকা|ফোস্কা|ভেসিকল", r"ফোসকা", "যন্ত্রণাদায়ক ফোসকা")
C("NAUSEA", "sym", r"বমি", r"বমি", "বমি বমি ভাব/বমি")
C("PHOTOPHOBIA", "sym", r"আলো[^;।,]{0,20}(অসহ্য|অস্বস্তি|সহ্য|কষ্ট)|ফটোফোবিয়া|আলোকাতর", r"আলোতে|আলো", "আলোতে অস্বস্তি")
C("CHRONIC_COUGH", "sym", r"দীর্ঘস্থায়ী কাশি|দীর্ঘদিন[^।]{0,15}কাশি|কাশি[^।]{0,250}(\d+ ?সপ্তাহ|\d+ ?মাস|\d+ ?বছর)", r"দীর্ঘস্থায়ী কাশি", "দীর্ঘস্থায়ী কাশি", sent=True)
C("WEIGHT_LOSS", "sym", r"ওজন (কমে|কমছে|কমেছে|হ্রাস)|ওজন কমা(?!ন)", r"ওজন কমা", "ওজন কমা")
C("NIGHT_SWEAT", "sym", r"রাতে[^;।,]{0,12}ঘাম", r"রাতে ঘাম", "রাতে ঘাম")
C("PROLONGED_FEVER", "sym", r"জ্বর[^।]{0,250}(\d+ ?সপ্তাহ|\d+ ?মাস|\b([7-9]|\d\d) ?দিন)|দীর্ঘ[^।]{0,15}জ্বর", r"দীর্ঘদিনের জ্বর", "দীর্ঘদিনের জ্বর (≥৭ দিন)", sent=True)
C("IRREG_MENSES", "sym", r"মাসিক[^;।,]{0,20}(অনিয়মিত|অনিয়ম|বিলম্ব|বন্ধ)|অনিয়মিত মাসিক|মাসিকের অনিয়ম|ঋতুস্রাব[^;।,]{0,15}অনিয়মিত", r"মাসিক অনিয়মিত|অনিয়মিত মাসিক", "মাসিক অনিয়মিত")
C("DYSURIA", "sym", r"প্রস্রাবে[^;।,]{0,10}জ্বালা|প্রস্রাব করতে জ্বালা|প্রস্রাবের সময় জ্বালা", r"প্রস্রাবে জ্বালা", "প্রস্রাবে জ্বালাপোড়া")
# --- drugs ('drug': mentioned in treatment, not negated)
C("ORS", "drug", r"ওআরএস|\bORS\b|ওরাল রিহাইড্রেশন", r"ওআরএস|ORS", "ওআরএস")
C("PPI", "drug", r"প্যান্টোপ্রাজল|ওমেপ্রাজল|এসোমেপ্রাজল|এসোমিপ্রাজল|র্যাবেপ্রাজল|রেবেপ্রাজল|ল্যানসোপ্রাজল|পিপিআই|\bPPI\b|প্রোটন পাম্প", r"PPI|প্রোটন পাম্প", "PPI")
C("ANTIHIST", "drug", r"সিটিরিজিন|লেভোসিটিরিজিন|ফেক্সোফেনাডিন|লোরাটাডিন|ডেসলোরাটাডিন|ফেনিরামিন|ক্লোরফেনিরামিন|হাইড্রক্সিজিন|বিলাস্টিন|রুপাটাডিন|অ্যান্টিহিস্টামিন", r"অ্যান্টিহিস্টামিন", "অ্যান্টিহিস্টামিন")
C("PERMETHRIN", "drug", r"পারমেথ্রিন|permethrin", r"পারমেথ্রিন|permethrin", "পারমেথ্রিন")
C("SSRI", "drug", r"এসসিটালোপ্রাম|এসিটালোপ্রাম|সিটালোপ্রাম|সার্ট্রালিন|ফ্লুঅক্সেটিন|ফ্লুওক্সেটিন|প্যারোক্সেটিন|ফ্লুভোক্সামিন|SSRI|এসএসআরআই", r"SSRI", "SSRI")
C("ATORVA", "drug", r"অ্যাটরভাস্টাটিন|অ্যাটোরভাস্টাটিন|atorvastatin", r"অ্যাটরভাস্টাটিন", "অ্যাটরভাস্টাটিন")
C("METFORMIN", "drug", r"মেটফর্মিন|মেটফরমিন|metformin", r"মেটফর্মিন|মেটফরমিন", "মেটফর্মিন")
C("AMOX", "drug", r"অ্যামোক্সিসিলিন|অ্যামক্সিসিলিন|amoxicillin", r"অ্যামোক্সিসিলিন", "অ্যামোক্সিসিলিন")
C("LOOP", "drug", r"ফুরোসেমাইড|ফিউরোসেমাইড|টরসেমাইড|বুমেটানাইড|লুপ ডাইইউরেটিক|ডাইইউরেটিক|furosemide|লাসিক্স", r"লুপ ডাইইউরেটিক|ডাইইউরেটিক", "লুপ ডাইইউরেটিক")
C("TAMSULOSIN", "drug", r"ট্যামসুলোসিন|টামসুলোসিন|tamsulosin", r"ট্যামসুলোসিন", "ট্যামসুলোসিন")
C("ACYCLOVIR", "drug", r"অ্যাসাইক্লোভির|অ্যাসিক্লোভির|ভ্যালাসাইক্লোভির|acyclovir", r"অ্যাসাইক্লোভির", "অ্যাসাইক্লোভির")
C("PARA_SYRUP", "drug", r"প্যারাসিটামল[^;।,]{0,30}সিরাপ|সিরাপ প্যারাসিটামল|প্যারাসিটামল সাসপেনশন", r"প্যারাসিটামল সিরাপ", "প্যারাসিটামল সিরাপ")
C("ANTIBIOTIC", "drug", ABX, r"অ্যান্টিবায়োটিক চলছে|অ্যান্টিবায়োটিক নিচ্ছ", "অ্যান্টিবায়োটিক")
# --- tests ('test': ordered or reported, not negated)
C("NS1", "test", r"NS1|এনএস১", r"NS1", "NS1 পরীক্ষা")
C("PLATELET", "test", r"প্লেটলেট", r"প্লেটলেট", "প্লেটলেট কাউন্ট")
C("BLOOD_TEST", "test", r"রক্ত(ের)? পরীক্ষা|\bCBC\b|রক্ত কালচার|ব্লাড কালচার|রক্তের কালচার", r"রক্ত পরীক্ষা", "রক্ত পরীক্ষা")
C("CT", "test", r"সিটি স্ক্যান|সি\.?টি\.? স্ক্যান|\bCT\b|CT scan", r"সিটি স্ক্যান", "সিটি স্ক্যান")
C("ECG_ECHO", "test", r"ইসিজি|\bECG\b|\bEKG\b|ইকোকার্ডিও|ইকো(?![ঀ-৿])|\becho\b", r"ইসিজি|ইকো", "ইসিজি/ইকো")
C("CXR", "test", r"বুকের এক্স-?রে|চেস্ট এক্স-?রে|chest x-?ray", r"বুকের এক্স-রে", "বুকের এক্স-রে")
C("URINE_RE", "test", r"প্রস্রাবের (রুটিন|পরীক্ষা|R/?E)|প্রস্রাব পরীক্ষা|urine R/?E|ইউরিন (আর/?ই|R/?E|রুটিন)|urine routine", r"urine R/E|ইউরিন", "urine R/E")
# --- raw statements (matched as written)
C("LIFESTYLE", "raw", r"জীবনযাপন (পরিবর্তন|পরিবর্তনের)|জীবনযাত্রার পরিবর্তন|লাইফস্টাইল", r"জীবনযাপন", "জীবনযাপন পরিবর্তনের পরামর্শ")
C("NO_ABX", "raw", r"অ্যান্টিবায়োটিক[^;।]{0,25}(প্রয়োজন নেই|প্রয়োজন নাই|দরকার নেই|লাগবে না|প্রয়োজন হয় না|খাবেন না)", r"অ্যান্টিবায়োটিকের প্রয়োজন নেই|প্রয়োজন নেই", "অ্যান্টিবায়োটিকের প্রয়োজন নেই বলা হয়েছে")
C("NSAID_AVOID", "raw", r"(NSAID|ব্যথানাশক|ইবুপ্রোফেন|অ্যাসপিরিন)[^;।]{0,45}(খাবেন না|খাওয়া যাবে না|সেবন করবেন না|এড়িয়ে|এড়ান|এড়াতে|নিষেধ|নয়(?![ঀ-৿]))", r"NSAID", "NSAID এড়িয়ে চলার পরামর্শ")
# --- functional (numeric / demographic)
FUNC_TRIG = {"HBA1C_GT9": r"HbA1c.{0,12}9", "URIC_HIGH": r"ইউরিক অ্যাসিড.{0,10}বেশি", "CHILD": r"বাচ্চা|শিশু", "ELDERLY": r"বয়স্ক", "MALE": r"পুরুষ", "FEMALE": r"মহিলা"}
FUNC_LABEL = {"HBA1C_GT9": "HbA1c ৯%-এর বেশি", "URIC_HIGH": "সিরাম ইউরিক অ্যাসিড বেশি", "CHILD": "শিশু", "ELDERLY": "বয়স্ক (৬০+)", "MALE": "পুরুষ", "FEMALE": "মহিলা"}
FUNC_TRIG = {k: re.compile(nrm(v), re.I) for k, v in FUNC_TRIG.items()}
AGE_RX = re.compile(r"(\d{1,3})\s*(?:বছর|Y/)"); MONTH_RX = re.compile(r"(\d{1,2})\s*মাস বয়সী")
def age_of(t):
    m = AGE_RX.search(t)
    if m: return int(m.group(1))
    return 0 if MONTH_RX.search(t) else None
def f_child(t, c):
    a = age_of(t); return (a is not None and a <= 12) or bool(re.search(r"(ছেলে|মেয়ে)\s*শিশু|শিশু রোগী|নবজাতক", t))
def f_elderly(t, c): a = age_of(t); return a is not None and a >= 60
def f_male(t, c): return bool(re.search(r"পুরুষ|Y/M\b|ছেলে", t))
def f_female(t, c): return bool(re.search(r"মহিলা|Y/F\b|মেয়ে|গর্ভবতী", t))
def f_hba1c(t, c):
    return any(float(v) > 9.0 for v in re.findall(r"(?:HbA1c|এইচবিএ১সি|এইচবিএ1সি)\D{0,12}(\d+(?:\.\d+)?)", t, re.I))
def f_uric(t, c):
    return any(float(v) > 6.8 for v in re.findall(r"ইউরিক অ্যাসিড\D{0,14}(\d+(?:\.\d+)?)", t))
FUNCS = {"HBA1C_GT9": f_hba1c, "URIC_HIGH": f_uric, "CHILD": f_child, "ELDERLY": f_elderly, "MALE": f_male, "FEMALE": f_female}

# --- clause analysis -------------------------------------------------------
NEG_RX = re.compile(L + r"(?:(?:নেই|নাই|নয়|না)(?![\u0980-\u09FF])|অনুপস্থিত|অস্বীকার|দেখা যায়নি|পাওয়া যায়নি|হয়নি|ছিল না)")
FAM_RX = re.compile(r"পরিবারে|বাবার|মায়ের|ভাইয়ের|বোনের|পারিবারিক|পরিবারের|Family")
ADV_RX = re.compile(r"পরামর্শ|Adv\b|উপদেশ|পরবর্তী|ফলোআপ|ফলো-আপ|F/U|পুনরায় সাক্ষাৎ|পুনরায় দেখা|রোগীকে|সতর্ক")
COND_RX = re.compile(r"হলে|যদি|এড়িয়ে|এড়ান|না হলে|বাড়লে|দেখা দিলে")
DX_RX = re.compile(r"নির্ণয়|Dx|ডায়াগনোসিস|ডায়াগনসিস|নিশ্চিত|শনাক্ত|সন্দেহ")
def analyse(text):
    t = nrm(text); out = []
    for si, sent in enumerate(re.split(r"[।]", t)):
        adv = bool(ADV_RX.search(sent)); dx = bool(DX_RX.search(sent))
        out.append(dict(t=sent.strip(), adv=adv, dx=dx, neg=False, fam=False, s=si, whole=True))
        for cl in re.split(r"[;,]", sent):
            if cl.strip(): out.append(dict(t=cl.strip(), adv=adv or bool(COND_RX.search(cl)), dx=dx, neg=bool(NEG_RX.search(cl)), fam=bool(FAM_RX.search(cl)), s=si))
    return t, out

def eval_concept(key, t, clauses):
    cd = CONCEPTS[key]; best, evid = 0, []
    for c in clauses:
        if c.get("whole", False) != cd["sent"]: continue
        if not cd["rx"].search(c["t"]): continue
        k = cd["kind"]
        if cd["sent"]: best = max(best, 1.5 if c["dx"] else 1); evid.append(c["s"]); continue
        if k in ("dx", "sym") and (c["neg"] or c["fam"] or c["adv"]): continue
        if k in ("drug", "test") and (c["neg"] or c["fam"]): continue
        s = 2 if (k == "dx" and c["dx"]) else (1.5 if (k in ("drug", "test") or (k == "sym" and c["dx"])) else 1)
        best = max(best, s); evid.append(c["s"])
    return best, evid

def parse_query(q):
    qn = nrm(q); keys = [k for k, cd in CONCEPTS.items() if cd["trig"] and cd["trig"].search(qn)]
    keys += [k for k, rx in FUNC_TRIG.items() if rx.search(qn)]
    kinds = dict(
        meds=bool(re.search(r"ওষুধ|চিকিৎসা|অ্যান্টিবায়োটিক দেওয়া|অ্যান্টিবায়োটিক .*বলুন", qn)),
        abx_only=bool(re.search(r"কোন কোন অ্যান্টিবায়োটিক", qn)),
        symptoms=bool(re.search(r"উপসর্গ উল্লেখ|কী কী উপসর্গ|উপসর্গ কী", qn)),
        findings=bool(re.search(r"ফাইন্ডিংস", qn)),
        test_results=bool(re.search(
            r"পরীক্ষার ফলাফল|পরীক্ষার রিপোর্ট|টেস্টের ফলাফল|রিপোর্ট কী|ফলাফল কী|পরীক্ষার ফল|test results?|lab results?",
            qn, re.I
        )),
        describe=bool(re.search(r"বর্ণনা দিন|বিস্তারিত", qn)))
    if re.search(r"কোন কোন অ্যান্টিবায়োটিক", qn) and "ANTIBIOTIC" in keys: keys.remove("ANTIBIOTIC")
    return keys, kinds
for _, r in qry_df.iterrows(): print(r.query_id, parse_query(r.query_text)[0])

q01 ['DM']
q02 ['PNEUMONIA']
q03 ['HF']
q04 ['ASTHMA', 'NIGHT_COUGH']
q05 ['GOUT', 'URIC_HIGH']
q06 ['DM', 'HTN']
q07 ['DYSPNEA']
q08 ['DIARRHEA', 'DEHYD', 'ORS', 'CHILD']
q09 ['DM']
q10 ['DM', 'HF', 'KIDNEY', 'LOOP']
q11 ['PEPTIC', 'EPIG_BURN', 'PPI', 'NSAID_AVOID']
q12 ['ALLERGIC_RHINITIS', 'SNEEZE', 'ANTIHIST']
q13 ['DENGUE', 'FEVER', 'NS1', 'PLATELET']
q14 ['SCABIES', 'NIGHT_ITCH', 'PERMETHRIN']
q15 ['DEPRESSION', 'FATIGUE', 'SSRI']
q16 ['SEPSIS', 'FEVER', 'CONFUSION', 'BLOOD_TEST', 'CT']
q17 ['HYPERCHOL', 'ATORVA', 'LIFESTYLE']
q18 ['ISCHEMIC', 'CHEST_PAIN', 'ECG_ECHO']
q19 ['DM', 'METFORMIN', 'HBA1C_GT9']
q20 ['PNEUMONIA', 'RTI', 'DYSPNEA', 'AMOX', 'CXR']
q21 ['URTICARIA']
q22 ['VIRAL_FEVER', 'FEVER', 'NO_ABX']
q23 ['BPH', 'TAMSULOSIN', 'ELDERLY', 'MALE']
q24 ['TB', 'CHRONIC_COUGH', 'WEIGHT_LOSS', 'NIGHT_SWEAT']
q25 ['FEVER', 'COLD', 'PARA_SYRUP', 'CHILD']
q26 ['ZOSTER', 'BLISTER', 'ACYCLOVIR']
q27 ['MIGRAINE', 'NAUSEA', 'PHOTOPHOBIA']
q28 ['TYPHOID', 'FEVER', 'PROLONGED_FEVER']


### PART 6.1 — RAG CORPUS, RETRIEVAL AND ANSWER GENERATION

In [13]:
from sklearn.metrics.pairwise import linear_kernel

class Note:
    def __init__(self, nid, text, ents):
        self.id, self.text, self.ents = str(nid), text, ents
        self.t, self.clauses = analyse(text)

def own_note(nid):
    toks, tags = own_docs[nid]
    ents = [(ty, " ".join(toks[a:b + 1])) for a, b, ty in tags_to_spans(tags)]
    return Note(nid, " ".join(toks), ents)

# RAG knowledge base: OWN ANNOTATED CORPUS ONLY.
own_notes = [own_note(nid) for nid in own_docs]
if not own_notes:
    raise ValueError("No own annotated notes were loaded; RAG cannot be built.")
assert len({n.id for n in own_notes}) == len(own_notes)

class Retriever:
    """
    Hybrid retrieval:
      - clause-aware concept evidence
      - partial concept coverage fallback (no all-or-nothing rejection)
      - character n-gram TF-IDF ranking
      - TEST_RESULT intent based on annotated TEST_RESULT/TEST entities
    """
    def __init__(self, notes):
        self.notes = notes
        self.tf = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 4),
            sublinear_tf=True, min_df=1
        )
        self.M = self.tf.fit_transform([n.t for n in notes])
        self.cache = {n.id: {} for n in notes}

    def concept(self, note, key):
        if key not in self.cache[note.id]:
            self.cache[note.id][key] = eval_concept(key, note.t, note.clauses)
        return self.cache[note.id][key]

    @staticmethod
    def _sentence_evidence(note, entity_types):
        sentences = note.text.split("।")
        wanted = [nrm(e) for ty, e in note.ents if ty in entity_types and e.strip()]
        return [
            i for i, sent in enumerate(sentences)
            if any(entity in nrm(sent) for entity in wanted)
        ][:5]

    def search(self, query, top_k=20):
        keys, kinds = parse_query(query)
        qn = nrm(query)
        sims = linear_kernel(self.tf.transform([qn]), self.M).ravel()
        candidates = []

        for i, note in enumerate(self.notes):
            lexical = float(sims[i])
            matched, evidence, concept_score = [], [], 0.0

            if kinds.get("test_results", False):
                result_entities = [e for ty, e in note.ents if ty == "TEST_RESULT" and e.strip()]
                test_entities = [e for ty, e in note.ents if ty == "TEST" and e.strip()]
                if result_entities:
                    matched.append("TEST_RESULT")
                    concept_score += 3.0
                if test_entities:
                    matched.append("TEST")
                    concept_score += 0.75
                if not matched:
                    continue
                evidence = self._sentence_evidence(note, {"TEST_RESULT", "TEST"})

            elif keys:
                for key in keys:
                    if key in FUNCS:
                        value = 1.0 if FUNCS[key](note.t, note.clauses) else 0.0
                        ev = []
                    else:
                        value, ev = self.concept(note, key)
                    if value > 0:
                        matched.append(key)
                        concept_score += float(value)
                        evidence.extend(ev)
                # Keep partial matches. Do not reject a note only because one requested
                # concept is absent; coverage determines its ranking position.
                if not matched:
                    continue

            else:
                # No recognised concept: lexical-only retrieval, but never return
                # zero-similarity notes as if they were evidence.
                if lexical <= 0.0:
                    continue

            coverage = (
                len(matched) / max(1, len(keys))
                if keys and not kinds.get("test_results", False)
                else 1.0
            )
            score = 2.0 * coverage + concept_score + lexical
            if kinds.get("test_results", False) and "TEST_RESULT" in matched:
                score += 2.0

            candidates.append({
                "score": float(score),
                "lexical_score": lexical,
                "coverage": float(coverage),
                "matched_concepts": matched,
                "note": note,
                "evidence": list(dict.fromkeys(evidence))
            })

        candidates.sort(
            key=lambda x: (x["coverage"], x["score"], x["lexical_score"]),
            reverse=True
        )
        total = len(candidates)
        return keys, kinds, candidates[:top_k], total

RET = {"own": Retriever(own_notes)}

def ents_of(note, entity_type):
    return [norm_ent(s) for ty, s in note.ents if ty == entity_type]

def count_notes(notes, entity_type, flt=None, top=12):
    counter = collections.Counter()
    for note in notes:
        entities = {
            x for x in ents_of(note, entity_type)
            if len(x) > 1 and not re.fullmatch(r"[\d.,/%\s]+", nrm(x))
        }
        for entity in entities:
            if flt is None or flt.search(nrm(entity)):
                counter[entity] += 1
    return counter.most_common(top)

def fmt_counts(items):
    return "; ".join(f"{name} ({bn(count)})" for name, count in items) if items else "নিজস্ব কর্পাসে উল্লেখ নেই"

SEX_M = re.compile(r"পুরুষ|Y/M\b|ছেলে")
SEX_F = re.compile(r"মহিলা|Y/F\b|মেয়ে")
DX_FALLBACK = re.compile(r"(?:Dx|নির্ণয়|ডায়াগনোসিস|ডায়াগনসিস)[^:]*:\s*([^।]+)")

def dx_sentences(note):
    raw = note.text.split("।")
    normalized = note.t.split("।")
    return [
        raw[i] for i in range(min(len(raw), len(normalized)))
        if DX_RX.search(normalized[i])
    ]

def one_liner(note):
    age = age_of(note.t)
    sex = "পুরুষ" if SEX_M.search(note.t) else ("মহিলা" if SEX_F.search(note.t) else "")
    who = ", ".join(x for x in [f"{bn(age)} বছর" if age is not None else "", sex] if x)
    diagnosis_text = " ".join(nrm(s) for s in dx_sentences(note))
    diagnoses = [
        e for e in dict.fromkeys(ents_of(note, "DIAGNOSIS"))
        if nrm(e) in diagnosis_text
    ]
    diagnosis = "; ".join(diagnoses[:2])
    if not diagnosis:
        match = DX_FALLBACK.search(note.text)
        diagnosis = match.group(1).strip() if match else ""
    medications = ", ".join(dict.fromkeys(ents_of(note, "MEDICATION")[:4]))
    results = ", ".join(dict.fromkeys(ents_of(note, "TEST_RESULT")[:4]))
    fields = [
        f"নির্ণয়: {diagnosis}" if diagnosis else "",
        f"ওষুধ: {medications}" if medications else "",
        f"পরীক্ষার ফলাফল: {results}" if results else ""
    ]
    return f"{note.id}: {who} — " + "; ".join(x for x in fields if x)

def make_answer(query, keys, kinds, records, total_matches):
    if not records:
        return (
            "নিজস্ব অ্যানোটেটেড কর্পাসে এই প্রশ্নের জন্য পর্যাপ্ত প্রাসঙ্গিক প্রমাণ পাওয়া যায়নি। "
            "এটি প্রমাণ করে না যে রোগীর এমন তথ্য নেই; শুধু উপলব্ধ কর্পাসে নির্ভরযোগ্য মিল পাওয়া যায়নি।"
        )

    notes = [record["note"] for record in records]
    lines = [f"নিজস্ব অ্যানোটেটেড কর্পাস থেকে {bn(total_matches)}টি প্রাসঙ্গিক নোট পাওয়া গেছে।"]

    if kinds.get("test_results"):
        lines.append("নথিভুক্ত পরীক্ষার ফলাফল: " + fmt_counts(count_notes(notes, "TEST_RESULT", top=20)) + "।")
        lines.append("সংশ্লিষ্ট পরীক্ষার নাম: " + fmt_counts(count_notes(notes, "TEST", top=15)) + "।")
    if kinds.get("symptoms"):
        lines.append("নথিভুক্ত উপসর্গ: " + fmt_counts(count_notes(notes, "SYMPTOM", top=15)) + "।")
    if kinds.get("findings"):
        lines.append("পরীক্ষা: " + fmt_counts(count_notes(notes, "TEST", top=12)) + "।")
        lines.append("পরীক্ষার ফলাফল: " + fmt_counts(count_notes(notes, "TEST_RESULT", top=12)) + "।")
        lines.append("সংশ্লিষ্ট উপসর্গ: " + fmt_counts(count_notes(notes, "SYMPTOM", top=10)) + "।")
    if kinds.get("meds"):
        antibiotic_pattern = re.compile(nrm(ABX), re.I)
        filter_pattern = antibiotic_pattern if kinds.get("abx_only") else None
        label = "নথিভুক্ত অ্যান্টিবায়োটিক" if kinds.get("abx_only") else "নথিভুক্ত ওষুধ"
        lines.append(label + ": " + fmt_counts(count_notes(notes, "MEDICATION", filter_pattern, top=15)) + "।")
    if kinds.get("describe"):
        lines.append("সাধারণ উপসর্গ: " + fmt_counts(count_notes(notes, "SYMPTOM", top=8)) + "।")
        lines.append("সাধারণ পরীক্ষা: " + fmt_counts(count_notes(notes, "TEST", top=8)) + "।")
        lines.append("সাধারণ ওষুধ: " + fmt_counts(count_notes(notes, "MEDICATION", top=8)) + "।")
        lines.append("নির্ণয়: " + fmt_counts(count_notes(notes, "DIAGNOSIS", top=8)) + "।")

    lines.append("প্রমাণ-সমর্থিত নোটের উদাহরণ:")
    lines.extend("• " + one_liner(note) for note in notes[:8])
    return "\n".join(lines)

def passage(note, evidence_indices=None, max_chars=600):
    sentences = note.text.split("।")
    if evidence_indices:
        selected = [
            sentences[i].strip() for i in evidence_indices
            if 0 <= i < len(sentences)
        ]
        if selected:
            return "। ".join(selected)[:max_chars]
    wanted = [
        nrm(entity) for ty, entity in note.ents
        if ty in {"TEST_RESULT", "TEST"} and entity.strip()
    ]
    selected = [
        sentence.strip() for sentence in sentences
        if any(entity in nrm(sentence) for entity in wanted)
    ]
    if selected:
        return "। ".join(selected)[:max_chars]
    return note.text[:max_chars]

# Run all fixed queries against the OWN corpus only.
out_rows = []
for row in qry_df.itertuples():
    keys, kinds, records, total = RET["own"].search(row.query_text, top_k=20)
    out_rows.append({
        "query_id": row.query_id,
        "query_text": row.query_text,
        "corpus": "own",
        "retrieved_note_ids": [r["note"].id for r in records],
        "retrieved_passages": [passage(r["note"], r["evidence"]) for r in records],
        "retrieval_scores": [round(r["score"], 4) for r in records],
        "lexical_scores": [round(r["lexical_score"], 4) for r in records],
        "concept_coverage": [round(r["coverage"], 4) for r in records],
        "matched_concepts": [r["matched_concepts"] for r in records],
        "answer": make_answer(row.query_text, keys, kinds, records, total),
        "n_matches": total
    })

with open(PRED / "rag_outputs.jsonl", "w", encoding="utf-8") as f:
    for output in out_rows:
        f.write(json.dumps(output, ensure_ascii=False) + "\n")

summary = pd.DataFrame(
    [(o["query_id"], o["n_matches"]) for o in out_rows],
    columns=["query_id", "own_matches"]
)
summary.to_csv(ANA / "rag_match_counts.csv", index=False)
print(summary.to_string(index=False))
print("\n--- sample test-result query ---")
sample = next((o for o in out_rows if "পরীক্ষার ফলাফল" in o["query_text"]), out_rows[0] if out_rows else None)
if sample:
    print(sample["query_text"])
    print(sample["answer"])


query_id  own_matches
     q01           24
     q02            7
     q03            1
     q04            5
     q05            1
     q06           33
     q07           19
     q08           40
     q09           24
     q10           29
     q11            7
     q12            2
     q13           42
     q14            1
     q15           20
     q16           57
     q17            3
     q18           22
     q19           26
     q20           41
     q21            0
     q22           41
     q23          149
     q24           11
     q25           65
     q26            4
     q27           27
     q28           41
     q29           47
     q30           48

--- sample test-result query ---
ডায়াবেটিস রোগীদের জন্য কোন কোন ওষুধ নথিভুক্ত করা আছে?
নিজস্ব অ্যানোটেটেড কর্পাস থেকে ২৪টি প্রাসঙ্গিক নোট পাওয়া গেছে।
নথিভুক্ত ওষুধ: ইনসুলিন (৪); গ্লিমেপিরাইড (২); মেটফরমিন (১); পটাশিয়াম ক্লোরাইড (১); প্রেগাবালিন (১); অ্যামোক্সিসিলিন-ক্লাভুলানেট (১); ফিউরোসেমাইড (১); ভ্যানকোমাইসিন 

## PART 7 — QA AND INTERACTIVE UI
Run the fixed Bengali query set, generate grounded answers, and inspect retrieved passages, note IDs, relevance scores and concept coverage through a dropdown interface.

In [14]:
# ---- PART 7: Fixed-query QA + interactive dropdown UI -----------------------
# The dropdown calls the SAME RET["own"] retrieval pipeline used above.

def run_medbhasha_query(query_text, top_k=10):
    keys, kinds, records, total = RET["own"].search(query_text, top_k=top_k)
    answer = make_answer(query_text, keys, kinds, records, total)
    evidence_rows = []
    for rank, record in enumerate(records, start=1):
        evidence_rows.append({
            "rank": rank,
            "note_id": record["note"].id,
            "score": round(record["score"], 4),
            "lexical_score": round(record["lexical_score"], 4),
            "concept_coverage": round(record["coverage"], 4),
            "matched_concepts": ", ".join(record["matched_concepts"]) or "lexical",
            "passage": passage(record["note"], record["evidence"])
        })
    return answer, pd.DataFrame(evidence_rows)

QUERY_OPTIONS = [
    (f"{row.query_id} — {row.query_text}", row.query_text)
    for row in qry_df.itertuples()
]
assert QUERY_OPTIONS, "No fixed queries found in test_queries_bn.csv"

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output

    query_dropdown = widgets.Dropdown(
        options=QUERY_OPTIONS,
        value=QUERY_OPTIONS[0][1],
        description="Query:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "70px"}
    )
    topk_slider = widgets.IntSlider(
        value=10, min=1, max=20, step=1,
        description="Top K:",
        continuous_update=False
    )
    run_button = widgets.Button(description="Run RAG query", button_style="primary", icon="search")
    output_area = widgets.Output()

    def _on_run_clicked(_):
        with output_area:
            clear_output(wait=True)
            query = query_dropdown.value
            answer, evidence_df = run_medbhasha_query(query, top_k=topk_slider.value)
            display(Markdown("## Selected Bengali query\n" + query))
            display(Markdown("## Grounded answer\n" + answer.replace("\n", "  \n")))
            display(Markdown("## Retrieved evidence"))
            if evidence_df.empty:
                display(Markdown(
                    "**No sufficiently relevant note was retrieved.** "
                    "This is not proof that the clinical fact is absent from the real world."
                ))
            else:
                display(evidence_df)

    run_button.on_click(_on_run_clicked)
    display(widgets.VBox([
        widgets.HTML("<h2>MedBhasha — Bengali Clinical RAG</h2>"),
        query_dropdown, topk_slider, run_button, output_area
    ]))
except Exception as ui_error:
    print("ipywidgets UI could not be rendered:", repr(ui_error))
    print("Use run_medbhasha_query(query_text) directly.")
    print(run_medbhasha_query(qry_df.iloc[0]["query_text"], top_k=5)[0])

qa_rows = []
for row in qry_df.itertuples():
    answer, evidence_df = run_medbhasha_query(row.query_text, top_k=10)
    qa_rows.append({
        "query_id": row.query_id,
        "query_text": row.query_text,
        "answer": answer,
        "retrieved_note_count": len(evidence_df),
        "retrieved_note_ids": ";".join(evidence_df.note_id.astype(str).tolist()) if len(evidence_df) else ""
    })
pd.DataFrame(qa_rows).to_csv(PRED / "qa_summary.csv", index=False, encoding="utf-8-sig")
print("Saved:", PRED / "qa_summary.csv")


Saved: /content/predictions/qa_summary.csv


,rank,note_id,score,lexical_score,concept_coverage,matched_concepts,passage
0,1,a006,4.2196,0.2196,1.0,"DM, HTN","সম্মানিত নেফ্রোলজিস্ট স্যার , ৫৮ বছর বয়সী পুর..."
1,2,f094,4.1955,0.1955,1.0,"DM, HTN","প্রি-অ্যানেসথেসিয়া নোট : ৫৮ বছর বয়সী নারী , ..."
2,3,b019,4.1773,0.1773,1.0,"DM, HTN","Referral : ৬২ বছর বয়সী মহিলা , পাঁচ বছর ধরে ড..."
3,4,a026,4.1344,0.1344,1.0,"DM, HTN",ডায়াবেটিস ও হাইপারটেনশন নিয়ে আগে থেকেই চিকিৎ...
4,5,a017,4.0512,0.0512,1.0,"DM, HTN","৬০ বছর বয়সী পুরুষ , hypertension ও diabetes এ..."
5,6,a001,3.1548,0.1548,0.5,DM,টাইপ-২ ডায়াবেটিস নিশ্চিত
6,7,b002,2.2325,0.2325,0.5,HTN,"পুরনো উচ্চ রক্তচাপ আছে , ডায়াবেটিস নেই"
7,8,f056,2.2124,0.2124,0.5,DM,"জরুরি বিভাগ : ৪১ বছর বয়সী পুরুষ , টাইপ-১ ডায়..."
8,9,a015,2.1901,0.1901,0.5,DM,"৫২ বছর বয়সী নারী , ডায়াবেটিস ও কিডনি রোগ নিয..."
9,10,f052,2.1827,0.1827,0.5,DM,"ডায়াবেটিস ক্লিনিক : ৬২ বছর বয়সী নারী , টাইপ-..."


## Retrieved evidence

,rank,note_id,score,lexical_score,concept_coverage,matched_concepts,passage
0,1,c008,8.3817,0.3817,1.0,"PNEUMONIA, RTI, DYSPNEA, AMOX, CXR",নিউমোনিয়া ধরে অ্যামোক্সিসিলিন সিরাপ দিনে দুইব...
1,2,c014,8.3820,0.2820,0.8,"PNEUMONIA, RTI, DYSPNEA, CXR","নবজাতকের নিউমোনিয়া সন্দেহ , তাই অ্যাম্পিসিলিন..."
2,3,f077,6.9644,0.3644,0.8,"PNEUMONIA, RTI, AMOX, CXR",নিউমোনিয়া ধরা হলো। অ্যামোক্সিসিলিন ওজন অনুযায...
3,4,c027,7.0136,0.3136,0.6,"PNEUMONIA, RTI, AMOX","শিশুর নিউমোনিয়া , community clinic : ১৪ মাস ব..."
4,5,d030,4.9981,0.2981,0.6,"PNEUMONIA, RTI, CXR","নিউমোনিয়া সম্ভাব্য , অ্যাজিথ্রোমাইসিন ৫০০ মিগ..."
5,6,d007,4.8667,0.1667,0.6,"PNEUMONIA, RTI, CXR",ছবিটি ভাইরাল নিউমোনিয়ার সঙ্গে সম্ভাব্য সংগতিপ...
6,7,a014,3.5088,0.2088,0.4,"DYSPNEA, CXR","গত এক সপ্তাহ ধরে কাশি ও শ্বাসকষ্ট বেড়েছে , জ্..."
7,8,f112,3.5011,0.2011,0.4,"DYSPNEA, CXR","পোস্ট-কোভিড ক্লিনিক : ৩৯ বছর বয়সী নারী , তিন ..."
8,9,f037,3.4542,0.1542,0.4,"DYSPNEA, CXR","কার্ডিওলজি ওয়ার্ড : ৭২ বছর বয়সী নারী , হার্ট..."
9,10,f146,2.9404,0.1404,0.4,"PNEUMONIA, RTI","হ্যান্ডওভার নোট : বেড ৪ , ৬৬ বছর বয়সী পুরুষ ,..."


## Retrieved evidence

,rank,note_id,score,lexical_score,concept_coverage,matched_concepts,passage
0,1,c030,5.7437,0.2437,0.75,"PCOS, IRREG_MENSES, FEMALE",পলিসিস্টিক ওভারি সিনড্রোম সন্দেহ। অনিয়মিত মাস...
1,2,c011,3.7142,0.2142,0.50,"IRREG_MENSES, METFORMIN","গাইনি OPD : ২৩ বছর বয়সী অবিবাহিত তরুণী , ছয় ..."
2,3,a001,2.1596,0.1596,0.25,METFORMIN,মেটফরমিন ৫০০ মিগ্রা দিনে দুইবার খাবারের পরে শু...
3,4,f045,2.1468,0.1468,0.25,METFORMIN,দীর্ঘদিন মেটফরমিন খাচ্ছেন
4,5,e030,2.1029,0.1029,0.25,METFORMIN,মেটফরমিন ৫০০ মিগ্রা দিনে দুইবার চলবে
5,6,a009,2.0508,0.0508,0.25,METFORMIN,Metformin 500 mg twice daily চলবে
6,7,c001,1.6218,0.1218,0.25,FEMALE,রক্তচাপ পরীক্ষা করে পাওয়া গেছে ১১০/৭০। হিমোগ্...
7,8,b017,1.6143,0.1143,0.25,FEMALE,"পোস্ট-অপ নোট । ৪২ বছর বয়সী মহিলা , ল্যাপারোস্..."
8,9,a023,1.6137,0.1137,0.25,FEMALE,হিমোগ্লোবিন ১০.৫ গ্রাম/ডেসিলিটার
9,10,a002,1.6099,0.1099,0.25,FEMALE,উচ্চ রক্তচাপ ফলো-আপ। আজ রক্তচাপ ১৫০/৯০ মিমি পা...


### PART 7.1 — VALIDATION, REPORT AND REPRODUCIBLE PACKAGING

In [15]:
# ---- format validation of all deliverables ---------------------------------
ner_chk = pd.read_csv(PRED / "ner_predictions.csv", encoding="utf-8", keep_default_na=False, dtype={"note_id": str})
txt = test_df.set_index("note_id").note_text
assert {"note_id", "start", "end", "entity_type"} <= set(ner_chk.columns)
assert ner_chk.entity_type.isin(ENTITY_TYPES).all() and (ner_chk.start >= 0).all() and (ner_chk.end > ner_chk.start).all()
assert all(e <= len(txt[n]) for n, e in zip(ner_chk.note_id, ner_chk.end))
assert all(txt[r.note_id][r.start:r.end] == r.entity_text for r in ner_chk.itertuples())
cl_chk = pd.read_csv(PRED / "clusters.csv", dtype={"note_id": str})
assert set(cl_chk.note_id) == set(test_df.note_id) and cl_chk.note_id.is_unique
rag_chk = [json.loads(l) for l in open(PRED / "rag_outputs.jsonl", encoding="utf-8")]
assert len(rag_chk) == len(qry_df), "Expected one own-corpus RAG output per fixed query"
assert all(len(r["retrieved_note_ids"]) <= 20 and r["answer"] for r in rag_chk)
assert all(r["corpus"] == "own" for r in rag_chk), "RAG must use own corpus only"
assert all(set(r["retrieved_note_ids"]) <= set(own_docs) for r in rag_chk)
assert all(len(r["retrieved_note_ids"]) == len(r["retrieval_scores"]) for r in rag_chk)
print("ALL FORMAT CHECKS PASSED:", len(ner_chk), "entities |", len(cl_chk), "cluster rows |", len(rag_chk), "RAG lines")

# ---- requirements.txt --------------------------------------------------------
from importlib import metadata
def _v(p):
    try: return f"{p}=={metadata.version(p)}"
    except Exception: return p
pkgs = ["numpy", "pandas", "scikit-learn", "torch", "transformers", "seqeval", "sklearn-crfsuite", "matplotlib", "networkx", "jupyter", "nbformat"]
(ROOT / "requirements.txt").write_text("\n".join(_v(p) for p in pkgs) + "\ngit+https://github.com/csebuetnlp/normalizer\n")

# ---- technical report (Markdown; convert to PDF with pandoc/Word/Docs) ----------
def _fmt(d): return "n/a" if not d else f"P={d['P']:.3f}  R={d['R']:.3f}  F1={d['F1']:.3f}"
top = lambda ty: ", ".join(freq[freq.entity_type == ty].head(8).e)
report = f"""# MedBhasha Round 2 — Technical Report ({TEAM_NAME})

## 1. Pipeline overview
Single notebook `notebooks/final_notebook.ipynb`: NER → clustering/analysis → RAG. Seed = {SEED}. No external APIs, no web retrieval. Test notes used exactly as released.

## 2. NER
* Model: **{'BanglaBERT (csebuetnlp/banglabert) token classification, fine-tuned' if USED_BACKEND == 'banglabert' else 'CRF (BanglaBERT unavailable in this run)'}**; BIO tags over {len(ENTITY_TYPES)} entity types; first-sub-word labelling; lr={LR}, batch={BATCH_SIZE}, AdamW + linear warm-up.
* Data: {len(own_docs)} own annotated notes ({own_df.shape[0]} tokens). Dev split {len(dev_ids)} notes (stratified by Round-1 id prefix); final model retrained on all notes for {best_epochs if best_epochs else 'CRF'} epochs.
* Dev (entity-level, exact match): BanglaBERT {_fmt(bert_scores)} | CRF baseline {_fmt(crf_scores)}.
* Offsets: tokeniser keeps (start,end) for every token; entity = first token start .. last token end on the unmodified `note_text` (code points, 0-based, end exclusive). All spans verified `note_text[start:end] == entity_text`.
* Output: {len(ner_chk)} entities in {ner_chk.note_id.nunique()} notes.
* Limitation: the Round-1 corpus is narrative prose while many test notes are in telegraphic `C/O … Dx: … Rx:` style, so domain shift is expected.

## 3. Clustering
* Features from NER output only (typed entity bag + word unigrams, TF-IDF, type weights Dx×3/Med×2/Sym×1.5), SVD({nsvd}), KMeans.
* k scanned 20–120; smallest k within 5 % of max Silhouette → **k = {BEST_K}**.
* **Silhouette (cosine) = {SIL:.4f}, Davies-Bouldin = {DBI:.4f}** (own data, see `predictions/analysis/cluster_k_selection.csv`).
* Hidden-label metrics (NMI/ARI) are computed by the organisers.

## 4. Frequency & relationship analysis (`predictions/analysis/`)
Entity frequency tables/charts per type, entity-type co-occurrence, diagnosis↔medication/symptom/test association tables (support, lift, PMI), diagnosis×medication heat-map, entity network, cluster profiles.
Top predicted diagnoses: {top('DIAGNOSIS')}. Top medications: {top('MEDICATION')}.

## 5. RAG and QA/UI
Retrieval-based (no fine-tuning). RAG indexes the team's own annotated corpus only. Hybrid retrieval uses clause-aware concept matching, partial concept coverage, and character n-gram TF-IDF. Generic test-result queries use annotated TEST_RESULT/TEST entities. Answers are extractive/aggregative and include retrieved note IDs, passages, scores, and concept coverage. The dropdown UI calls the same retrieval function used for the saved fixed-query outputs.

## 6. Reproducibility
`requirements.txt`, fixed seeds, deterministic KMeans/SVD, no network use except downloading the public BanglaBERT checkpoint and normaliser.
"""
(ROOT / "REPORT.md").write_text(report, encoding="utf-8")
(ROOT / "README.md").write_text(f"""# {TEAM_NAME} — MedBhasha Round 2
1. Put the data CSVs in `data/` (annotated corpus, test notes, test queries).
2. `pip install -r requirements.txt`; open `notebooks/final_notebook.ipynb`; *Run all* (GPU recommended).
3. Outputs: `predictions/` (ner_predictions.csv, clusters.csv, rag_outputs.jsonl, analysis/).
Round-1 artefacts (raw_data.csv, validation_report.txt, source declaration, EDA summary) go in `round1/`. Technical report: `REPORT.md` -> export to PDF.
""", encoding="utf-8")

# ---- zip (test notes are NOT redistributed) --------------------------------------
ZIP = ROOT.parent / f"{TEAM_NAME}_MedBhasha.zip" if (ROOT / "notebooks").exists() else Path(f"{TEAM_NAME}_MedBhasha.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        rel = p.relative_to(ROOT)
        if p.is_dir() or rel.parts[0] in (".git", "__pycache__", "src") or "test_data" in p.name or "clinical_notes" in p.name or p.suffix == ".zip": continue
        z.write(p, Path(f"{TEAM_NAME}_MedBhasha") / rel)
print("packaged ->", ZIP, "| remember: add round1/ files (raw_data.csv, validation_report.txt, source declaration, EDA summary) and export REPORT.md to PDF")

ALL FORMAT CHECKS PASSED: 50686 entities | 1691 cluster rows | 30 RAG lines
packaged -> Train Test Panic_MedBhasha.zip | remember: add round1/ files (raw_data.csv, validation_report.txt, source declaration, EDA summary) and export REPORT.md to PDF
